$$
\newcommand{\dd}{\text{d}}
\newcommand{\pdv}[2]{ \frac{\partial #1}{\partial #2}  }
\newcommand{\dv}[2]{ \frac{\dd #1}{\dd #2} }
$$

# Bistable dynamics with colored noise

__Jared Callaham (2020)__

Supercritical pitchfork normal form forced by colored noise (Ornstein-Uhlenbeck process):
\begin{align}
\dot{x} &= \lambda x - \mu x^3 + v\\
\dot{v} &= -\alpha v + w,
\end{align}
where $w$ are still white noise processes.  To keep the noise strength comparable to an equivalent white noise process, the amplitude of the white noise process will scale with $\alpha$.

To explore separation of scales, we can do a long simulation where
$$ (\Delta t)^{-1} \gg \alpha \gg \lambda. $$
That is, the time step of simulation is finer than the decorrelation time, which is in turn finer than the time scale of the drift dynamics.  So for now
$$ (\Delta t)^{-1} = 10^3, \hspace{1cm} \alpha = 10^2, \hspace{1cm} \lambda = 1.$$
But note that once these statistics are established, we might be able to get away with time-stepping on the scale of the decorrelation time, i.e.
$$ (\Delta t)^{-1} \sim \alpha \gg \lambda. $$
Once we have this data we can explore the effect of sampling frequency, aiming to establish a dual separation of scales between the forcing and the dominant dynamics.

In [2]:
import numpy as np

thing = np.zeros((5, 5))
thing[3, 4:] = 1
thing[2:, 1] = 1
print(thing.astype(bool))

[[False False False False False]
 [False False False False False]
 [False  True False False False]
 [False  True False False  True]
 [False  True False False False]]


In [ ]:
import h5py

# Numpy
import numpy as np
from numpy.linalg import lstsq
import numpy.random as rng

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rcParams
mpl.rc('text', usetex=True)
mpl.rc('font', family='serif')
mpl.rc('xtick', labelsize=14)
mpl.rc('ytick', labelsize=14)
mpl.rc('axes', labelsize=20)
mpl.rc('axes', titlesize=20)
mpl.rc('figure', figsize=(6, 4))
%config InlineBackend.figure_format = 'retina'
mpl_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
              '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
              '#bcbd22', '#17becf']

# Scipy
from scipy.optimize import curve_fit, minimize
from scipy.signal import welch
from scipy import sparse, linalg

import sympy

# Custom packages
import utils
import fpsolve

ModuleNotFoundError: No module named 'h5py'

### Simulate model (SDE runs in Julia)

In [ ]:
# !julia ./sim_pitchfork.jl

In [ ]:
data = h5py.File("./data/pitchfork.mat", "r")
X = np.array(data["X"])
dt = data["dt"][0]
lamb = data["lamb"][0]
mu = data["mu"][0]

t = dt * np.arange(len(X))
f = lambda x, lamb, mu: lamb * x - mu * x**3

In [ ]:
# Plot solution
plt.figure(figsize=(10, 2))

plot_time = 500
plt.plot(t[t < plot_time], X[t < plot_time], "k")
plt.grid()
plt.xlabel("$t$", fontsize=24)
plt.ylabel("$x$", fontsize=24)

plt.show()

# Kramers-Moyal average

To see the effect of coarse vs fine sampling we can compute the Kramers-Moyal average at two different sampling rates.  As can be seen in the following figure, the correlations in the noise tend to destroy the Kramers-Moyal average.

The KM_avg function is in `utils.py`

Note that there is no "truth" for diffusion, since the model is really driven by a different dynamical process.  The coarse sampling has more or less the right shape, but is distorted by the finite-time effects

In [ ]:
# Plot truth, fast sampling, slow sampling

N = 32  # Number of bins
bins = np.linspace(-1.5, 1.5, N + 1)
h = bins[1] - bins[0]
centers = (bins[:-1] + bins[1:]) / 2

f_fine, a_fine, _, _ = utils.KM_avg(X, bins, stride=1, dt=dt)
f_coarse, a_coarse, _, _ = utils.KM_avg(X, bins, stride=500, dt=dt)

plt.figure(figsize=(10, 4))
plt.subplot(121)
plt.plot(centers, f_fine, ".", markersize=10, c=mpl_colors[0], label=r"$\tau=10^{-3}$")
plt.plot(centers, f_coarse, ".", markersize=10, c=mpl_colors[1], label=r"$\tau=0.5$")
plt.plot(centers, f(centers, lamb, mu), "k--", lw=3, label="Truth")
plt.legend(fontsize=14)
plt.title("Drift")
plt.xlabel("$x$", fontsize=24)
plt.ylabel("$f(x)$", fontsize=24)
plt.grid()

plt.subplot(122)
plt.plot(centers, a_fine, ".", c=mpl_colors[0], markersize=10)
plt.plot(centers, a_coarse, ".", c=mpl_colors[1], markersize=10)
plt.title("Diffusion")
plt.xlabel("$x$", fontsize=24)
plt.ylabel("$a(x)$", fontsize=24)
plt.grid()

plt.subplots_adjust(wspace=0.3)
plt.show()

# Adjoint optimization

Correcting the finite-time distortion with the adjoint Fokker-Planck equation.  The routines to call the optimizer and the cost function are in `utils.py`.

In [ ]:
## Kramers-Moyal average

N = 32  # Number of bins
bins = np.linspace(-2, 2, N + 1)
dx = bins[1] - bins[0]
centers = (bins[:-1] + bins[1:]) / 2

stride = 500  # How many snapshots to skip
f_KM, a_KM, f_err, a_err = KM_avg(X, bins, stride=stride, dt=dt)

In [ ]:
### Build SINDy libraries with sympy
x = sympy.symbols("x")

f_expr = np.array([x**i for i in np.arange(4)])  # Polynomial library for drift
s_expr = np.array([x**i for i in np.arange(3)])  # Polynomial library for diffusion

# Convert sympy expressions into library matrices
lib_f = np.zeros([len(f_expr), N])
for k in range(len(f_expr)):
    lamb_expr = sympy.lambdify(x, f_expr[k])
    lib_f[k] = lamb_expr(centers)

lib_s = np.zeros([len(s_expr), N])
for k in range(len(s_expr)):
    lamb_expr = sympy.lambdify(x, s_expr[k])
    lib_s[k] = lamb_expr(centers)

In [ ]:
# Initialize Xi with least squares regression (no finite-time corrections)

Xi0 = np.zeros((len(f_expr) + len(s_expr)))
mask = np.nonzero(np.isfinite(f_KM))[0]
Xi0[: len(f_expr)] = lstsq(lib_f[:, mask].T, f_KM[mask], rcond=None)[
    0
]  # Regression against drift
Xi0[len(f_expr) :] = lstsq(lib_s[:, mask].T, np.sqrt(2 * a_KM[mask]), rcond=None)[
    0
]  # Regression against diffusion

print(Xi0)

#### SSR for model selection

In [ ]:
### Weights: uncertainties in Kramers-Moyal
# This is helpful, but not that critical.  The specific choice of weights doesn't matter that much
W = np.array((f_err.flatten(), a_err.flatten()))
W[np.less(abs(W), 1e-12, where=np.isfinite(W))] = (
    1e6  # Set zero entries to large weights
)
W[np.logical_not(np.isfinite(W))] = (
    1e6  # Set NaN entries to large numbers (small weights)
)
W = 1 / W  # Invert error for weights
W = W / np.nansum(W.flatten())

# Compute empirical PDF
p_hist = np.histogram(X, bins, density=True)[0]

# Initialize adjoint solver
afp = fpsolve.AdjFP(centers)

# Initialize forward steady-state solver
fp = fpsolve.SteadyFP(N, dx)

# Optimization parameters
params = {
    "W": W,
    "f_KM": f_KM,
    "a_KM": a_KM,
    "Xi0": Xi0,
    "f_expr": f_expr,
    "s_expr": s_expr,
    "lib_f": lib_f.T,
    "lib_s": lib_s.T,
    "N": N,
    "kl_reg": 10,
    "fp": fp,
    "afp": afp,
    "p_hist": p_hist,
    "tau": stride * dt,
    "radial": False,
}

# Use anonymous function to automatically pass the cost function
opt_fun = lambda params: utils.AFP_opt(utils.cost, params)
Xi, V = utils.SSR_loop(opt_fun, params)

In [ ]:
####################
# SSR cost function
####################

labels = [r"${0}$".format(sympy.latex(t)) for t in np.concatenate((f_expr, s_expr))]

active = abs(Xi) > 1e-8
n_terms = len(labels)
plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.scatter(np.arange(len(V)), V, c="k")
plt.gca().set_xticks(np.arange(n_terms - 1))
plt.gca().set_xticklabels(np.arange(n_terms, 1, -1))
plt.xlabel("Sparsity")
plt.ylabel(r"Cost")
# plt.gca().set_yscale('log')
plt.grid()

plt.subplot(122)
plt.pcolor(active, cmap="bone_r", edgecolors="gray")
plt.gca().set_yticks(0.5 + np.arange(n_terms))
plt.gca().set_yticklabels(labels)
plt.gca().set_xticks(0.5 + np.arange(n_terms - 1))
plt.gca().set_xticklabels(np.arange(n_terms, 1, -1))
plt.xlabel("Sparsity")
plt.ylabel("Active terms")
plt.show()

In [ ]:
# Select model with the fewest terms before the cost function spikes
n_terms = 3
print(Xi[:, 1 - n_terms])
print(V[1 - n_terms])
Xi_f = Xi[: len(f_expr), 1 - n_terms]
Xi_s = Xi[len(f_expr) :, 1 - n_terms]

# Functions from the expressions
f_sindy = sympy.lambdify(x, utils.sindy_model(Xi_f, f_expr))
a_sindy = sympy.lambdify(x, 0.5 * utils.sindy_model(Xi_s, s_expr) ** 2)

f_vals = f_sindy(centers)
a_vals = a_sindy(centers)

# Check if a scalar (happens when library is a constant)
if np.isscalar(a_vals):
    a_vals = a_vals + 0 * centers
if np.isscalar(f_vals):
    f_vals = f_vals + 0 * centers

In [ ]:
# Compare PDFs: empirical vs Fokker-Planck solution with model

p_fit = fp.solve(f_vals, a_vals)
print(
    "KL divergence (LINDy model): {0:0.5f}".format(
        utils.kl_divergence(p_hist, p_fit, dx=dx, tol=1e-6)
    )
)

plt.figure(figsize=(4, 2))
plt.plot(centers, p_hist, "k", label="Data", lw=3)
plt.plot(centers, p_fit, "--", c=mpl_colors[1], label="Model", lw=3)
plt.legend(fontsize=14)
# plt.gca().set_yscale('log')
# plt.xlim([0, 3])
plt.xlabel("$x$", fontsize=24)
plt.ylabel("$p(x)$", fontsize=24)
plt.grid()

#### Predicted finite-time evolution of Kramers-Moyal coefficients

In particular, we see how the apparent state-dependent diffusion can arise from constant diffusion and coarse sampling

In [ ]:
afp.precompute_operator(f_vals, a_vals)
f_tau, a_tau = afp.solve(stride * dt)

plt.figure(figsize=(10, 3))
plt.subplot(121)
plt.plot(centers, f(centers, lamb, mu), c="gray", lw=2, label="True drift")
plt.errorbar(centers, f_KM, f_err, ls="", marker=".", markersize=8, c="k")
plt.plot(centers, f_vals, "k", lw=2)
plt.plot(centers, f_tau, "k:", lw=2)
plt.legend(fontsize=14)
plt.title("Drift")
plt.xlabel("$x$", fontsize=24)
plt.ylabel("$f(x)$", fontsize=24)
plt.grid()
plt.xlim([-1.8, 1.8])
plt.ylim([-3, 3])


plt.subplot(122)
plt.errorbar(
    centers,
    a_KM,
    a_err,
    ls="",
    marker=".",
    markersize=8,
    c="k",
    label=r"K-M  ($\tau = 0.5$)",
)
plt.plot(centers, a_vals, "k", lw=2, label=r"Model ($\tau = 0$)")
plt.plot(centers, a_tau, "k:", lw=2, label=r"Model ($\tau = 0.5$)")
plt.legend(fontsize=14)
plt.title("Diffusion")
plt.xlabel("$x$", fontsize=24)
plt.ylabel("$a(x)$", fontsize=24)
plt.grid()
plt.xlim([-1.8, 1.8])
plt.ylim([0, 0.5])

plt.subplots_adjust(wspace=0.3)
plt.show()